In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="CalculateMoreVariables", dataName="TKE",
                                dtype='float32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#Calculation Functions
def GetInputVariables(inputDataDirectory, timeString):
    u_data = DataManager.GetTimestepData(inputDataDirectory, timeString, variableName="uinterp")
    v_data = DataManager.GetTimestepData(inputDataDirectory, timeString, variableName="vinterp")
    w_data = DataManager.GetTimestepData(inputDataDirectory, timeString, variableName="winterp")

    inputDictionary = {'u_data': u_data,
                       'v_data': v_data,
                       'w_data': w_data
                      }
    return inputDictionary

def ComputeTKE(u,v,w):
    # Inputs shape: (z, y, x)
    [u_pert] = ComputePerturbation(u)
    [v_pert] = ComputePerturbation(v)
    [w_pert] = ComputePerturbation(w)

    TKE = 0.5 * (u_pert**2 + v_pert**2 + w_pert**2)
    return [TKE]

def ComputePerturbation(field):
    fieldMean = np.mean(field, axis=(1, 2), keepdims=True)
    perturbation = field - fieldMean
    return [perturbation]

def VariableCalculation(inputDictionary, t):
    [u_data,v_data,w_data] = (inputDictionary[k] for k in ['u_data','v_data','w_data'])

    #calculating TKE
    [TKE] = ComputeTKE(u_data,v_data,w_data)

    outputDictionary = {
        'TKE': TKE
    }
    return outputDictionary

In [ ]:
####################################
#RUNNING

In [ ]:
#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
for t in tqdm(loop_elements, total=len(loop_elements)):
    if np.mod(t,1)==0: print(f'Current time {t}')

    #getting timestring for loading input data
    timeString = ModelData.timeStrings[t]

    #loading input variables
    inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString)

    #calculating variables
    outputDictionary = VariableCalculation(inputDictionary, t)
    
    #outputting
    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary)

In [ ]:
####################################
#TESTING

In [ ]:
# #testing single timestep
# def ComputeMeanProfile(field, axis):
#     meanProfile = np.mean(field, axis=axis)
#     return [meanProfile]

# def PlotTkeProfiles(TKE, zHeight, coastalIndex=180, inlandIndex=350):
#     [TkeZ_Domain] = ComputeMeanProfile(TKE, axis=(1, 2))
#     [TkeZ_Coastal_25km] = ComputeMeanProfile(TKE[:, :, coastalIndex], axis=1)
#     [TkeZ_Inland] = ComputeMeanProfile(TKE[:, :, inlandIndex], axis=1)

#     fig, ax = plt.subplots()
#     ax.plot(TkeZ_Domain, zHeight, label='Domain mean', color='k')
#     ax.plot(TkeZ_Coastal_25km, zHeight, label='Coastal+25km', color='blue')
#     ax.plot(TkeZ_Inland, zHeight, label='Inland',color='green')

#     ax.set_xlabel('TKE ' + r'$(m^2/s^2)$')
#     ax.set_ylabel('z (km)')
#     ax.set_ylim(0, 6)
#     ax.legend()

#     return [fig, ax]

# t=100
# timeString = ModelData.timeStrings[t]
# inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString)

# #calculating variables
# outputDictionary = VariableCalculation(inputDictionary, t)
# TKE=outputDictionary['TKE']

# [Fig, Ax] = PlotTkeProfiles(TKE, ModelData.zh)
# plt.show()